In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

In [3]:
img_size = 224
batch_size = 24

train_dir = 'data_tomat3/train'
val_dir = 'data_tomat3/valid'

# train_datagen = ImageDataGenerator(
#     rescale=1./255,
#     rotation_range=20,
#     zoom_range=0.2,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     horizontal_flip=True
# )

# # train_datagen = ImageDataGenerator(rescale=1./255)


# val_datagen = ImageDataGenerator(rescale=1./255)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_dataset = train_datagen.flow_from_directory(train_dir,
    target_size= (224, 224),
    batch_size = 24,
    class_mode='categorical'
)

val_dataset = val_datagen.flow_from_directory(val_dir,
    target_size= (224, 224),
    batch_size = 24,
    class_mode='categorical',
    shuffle=False                            
)

num_classes = train_dataset.num_classes

Found 10000 images belonging to 10 classes.
Found 984 images belonging to 10 classes.


In [4]:
base_model = MobileNetV3Large(
    input_shape=(img_size, img_size, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # freeze

In [5]:
import tensorflow_addons as tfa

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)
output = Dense(num_classes, activation='softmax')(x)


model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    # loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    loss = tfa.losses.SigmoidFocalCrossEntropy(),
    metrics=['accuracy', 
            tf.keras.metrics.TopKCategoricalAccuracy(k=3)]
)

model.summary()

C:\Users\Temp\anaconda3\envs\mynew_env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
C:\Users\Temp\anaconda3\envs\mynew_env\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 rescaling (Rescaling)          (None, 224, 224, 3)  0           ['input_1[0][0]']                
                                                                                                  
 Conv (Conv2D)                  (None, 112, 112, 16  432         ['rescaling[0][0]']              
                                )                                                                 
                                                                                              

In [6]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

Epoch 1/10
417/417 [==============================] - 448s 1s/step - loss: 0.2760 - accuracy: 0.5572 - top_k_categorical_accuracy: 0.8211 - val_loss: 0.1065 - val_accuracy: 0.8303 - val_top_k_categorical_accuracy: 0.9736
Epoch 2/10
417/417 [==============================] - 310s 743ms/step - loss: 0.1306 - accuracy: 0.7771 - top_k_categorical_accuracy: 0.9566 - val_loss: 0.0895 - val_accuracy: 0.8486 - val_top_k_categorical_accuracy: 0.9715
Epoch 3/10
417/417 [==============================] - 311s 746ms/step - loss: 0.1056 - accuracy: 0.8212 - top_k_categorical_accuracy: 0.9692 - val_loss: 0.0729 - val_accuracy: 0.8740 - val_top_k_categorical_accuracy: 0.9766
Epoch 4/10
417/417 [==============================] - 335s 803ms/step - loss: 0.0985 - accuracy: 0.8323 - top_k_categorical_accuracy: 0.9747 - val_loss: 0.0740 - val_accuracy: 0.8811 - val_top_k_categorical_accuracy: 0.9817
Epoch 5/10
417/417 [==============================] - 331s 793ms/step - loss: 0.0881 - accuracy: 0.8500 - t

In [ ]:
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    # loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    loss = tfa.losses.SigmoidFocalCrossEntropy(),
    metrics=['accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=3)]
)

history_finetune = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)


# base_model.trainable = True  # buka seluruh layer

# model.compile(
#     optimizer=Adam(learning_rate=1e-5),
#     loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
#     metrics=['accuracy']
# )

# history_finetune = model.fit(
#     train_dataset,
#     validation_data=val_dataset,
#     epochs=5
# )

Epoch 1/10
166/417 [==========>...................] - ETA: 3:19 - loss: 0.5525 - accuracy: 0.5449 - top_k_categorical_accuracy: 0.8406

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

# def predict_leaf(img_path):
#     img = image.load_img(img_path, target_size=(img_size, img_size))
#     img = image.img_to_array(img) / 255.0
#     img = np.expand_dims(img, axis=0)

#     pred = model.predict(img)
#     class_index = np.argmax(pred)
#     class_label = list(train_dataset.class_indices.keys())[class_index]
#     confidence = pred[0][class_index]

#     return class_label, confidence

def predict_leaf(img_path):
    img = image.load_img(img_path, target_size=(img_size, img_size))
    img = image.img_to_array(img)
    img = preprocess_input(img)         # FIXED!
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)
    class_index = np.argmax(pred)
    class_label = list(train_dataset.class_indices.keys())[class_index]
    confidence = pred[0][class_index]

    return class_label, confidence

label, conf = predict_leaf("data_tomat2/valid/Tomato_mosaic_virus/2afd76a3-68f8-42cf-a68a-2f3076168dc0___PSU_CG 2260_90deg.jpg")
print(f"Prediksi: {label} | Confidence: {conf:.2f}")

In [9]:
model.save("Aaron_ModelKlasifikasi3.h5")

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()
plt.plot(history.history['loss'], color='teal', label='loss')
plt.plot(history.history['val_loss'], color='orange', label='val_loss')
fig.suptitle('Loss', fontsize=20)
plt.legend(loc="upper left")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(history.history['top_k_categorical_accuracy'], color='teal', label='sparse_categorical_accuracy')
plt.plot(history.history['val_top_k_categorical_accuracy'], color='orange', label='val_sparse_categorical_accuracy')
fig.suptitle('Accuracy', fontsize=20)
plt.legend(loc="upper left")
plt.show()